In [26]:
import itk
import numpy as np
from pathlib import Path
from typing import Union
import os

In [36]:
BASE_PATH = Path(os.getcwd())
input_file = BASE_PATH / "inputs/Panoramix-cropped.nii.gz"
input_file = BASE_PATH / "inputs/salt_pepper2.png"

In [30]:
itk_image = itk.imread(str(input_file))
np_image = itk.GetArrayFromImage(itk_image)

size = itk.size(itk_image)


In [48]:
# Leer la imagen
pixel_type = itk.ctype("unsigned char")
image_type = itk.Image[pixel_type, 2]

reader = itk.ImageFileReader[image_type].New()
reader.SetFileName(str(input_file))
reader.Update()

itk_image = reader.GetOutput()
size = itk.size(itk_image)
np_image = itk.GetArrayFromImage(itk_image)

In [47]:
# Import the necessary libraries
import itk

# Specify the pixel type and image type
pixel_type = itk.ctype("unsigned char")
image_type = itk.Image[pixel_type, 2]

# Set up the reader and provide the input file name
reader = itk.ImageFileReader[image_type].New()
reader.SetFileName(str(input_file))

# Update the reader to ensure the pipeline is executed
reader.Update()

# Get the ITK image from the reader
itk_image = reader.GetOutput()

# Get the size of the image (optional, to verify dimensions)
size = itk.size(itk_image)
print(f"Image size: {size}")

# Convert the ITK image to a NumPy array
np_image = itk.GetArrayFromImage(itk_image)

# Print or use the NumPy array
print(np_image)

Image size: itkSize2 ([361, 354])
[[255 231 130 ... 252 254 255]
 [255 230 123 ... 251 254 255]
 [255 230 124 ... 251 254 255]
 ...
 [255 240 175 ... 243 252 255]
 [255 240 178 ... 243 252 255]
 [255 241 180 ... 243 252 255]]


In [49]:
size

itkSize2 ([361, 354])

In [50]:
np_image.shape

(354, 361)

In [31]:
size

itkSize2 ([361, 354])

In [32]:
np_image.shape

(354, 361, 4)

In [35]:
np_image[:,:,2]

array([[255, 231, 130, ..., 252, 254, 255],
       [255, 230, 123, ..., 251, 254, 255],
       [255, 230, 124, ..., 251, 254, 255],
       ...,
       [255, 240, 175, ..., 243, 252, 255],
       [255, 240, 178, ..., 243, 252, 255],
       [255, 241, 180, ..., 243, 252, 255]], shape=(354, 361), dtype=uint8)

In [18]:
def adaptive_median_filter(
    input_file: Union[str, Path],
    output_file: Union[str, Path],
    max_radius: int
) -> None:
    itk_image = itk.imread(str(input_file))

    np_image = itk.GetArrayFromImage(itk_image)
    
    depth, height, width = np_image.shape
    output_image = np.copy(np_image)

    for z in range(depth):
        for y in range(height):
            for x in range(width):
                output_image[z, y, x] = process_pixel(np_image, x, y, z, max_radius)

    itk_output = itk.GetImageFromArray(output_image)
    itk_output.SetSpacing(itk_image.GetSpacing())  # Mantener la misma información espacial
    itk_output.SetOrigin(itk_image.GetOrigin())

    itk.imwrite(itk_output, str(output_file))

    return np_image, output_image


def process_pixel(image: np.ndarray, x: int, y: int, z: int, max_radius: int) -> int:
    """ Aplica el filtro de mediana adaptativo a un solo píxel """
    radius = 3
    while radius <= max_radius:
        window = get_window(image, x, y, z, radius)
        min_val, med_val, max_val = np.min(window), np.median(window), np.max(window)

        # Nivel 1: Evaluar la mediana
        if min_val < med_val < max_val:
            # Nivel 2: Evaluar el valor del píxel original
            pixel_value = image[z, y, x]
            if min_val < pixel_value < max_val:
                return pixel_value
            else:
                return med_val
        else:
            radius += 1  # Aumentar el radio

    return image[z, y, x]  # Si se supera max_radius, devolver el píxel original


def get_window(image: np.ndarray, x: int, y: int, z: int, radius: int) -> np.ndarray:
    """ Extrae la ventana 3d, se le suma uno al máximo dado que el slicer de numpy no incluye el límite superior"""
    z_min, z_max = max(0, z - radius), min(image.shape[0], z + radius + 1)
    y_min, y_max = max(0, y - radius), min(image.shape[1], y + radius + 1)
    x_min, x_max = max(0, x - radius), min(image.shape[2], x + radius + 1)
    
    return image[z_min:z_max, y_min:y_max, x_min:x_max]


# Ejemplo de uso
input_array, output_array = adaptive_median_filter(BASE_PATH / "inputs/9_percent_noise.nii.gz", "output.nii.gz", max_radius=10)


In [12]:
input_array[100,0:217,0:181].shape

(217, 181)

In [13]:
output_array[100,0:217,0:181].shape

(217, 181)

In [22]:
input_array[100,0:217,0:181]

array([[ 33.70965 , 141.99646 ,  78.29833 , ..., 106.12436 ,  69.2465  ,
         26.669329],
       [ 35.050663,  55.836365,  69.58175 , ...,  89.026436,  53.824844,
         65.89397 ],
       [ 65.89397 ,  78.29833 ,  13.929705, ..., 203.68306 ,  35.050663,
        128.25107 ],
       ...,
       [101.76607 ,  53.15434 , 153.05981 , ...,  15.941224,  46.784527,
        102.77183 ],
       [ 31.027622,  40.749966,  45.443512, ...,  73.94004 ,  26.334076,
         78.63359 ],
       [154.06558 , 117.85822 , 176.19229 , ..., 112.82942 , 118.193474,
         88.02068 ]], shape=(217, 181), dtype=float32)

In [23]:
output_array[100,0:217,0:181]

array([[ 33.70965 , 141.99646 ,  78.29833 , ..., 106.12436 ,  69.2465  ,
         26.669329],
       [ 35.050663,  55.836365,  69.58175 , ...,  89.026436,  53.824844,
         65.89397 ],
       [ 65.89397 ,  78.29833 ,  13.929705, ..., 203.68306 ,  35.050663,
        128.25107 ],
       ...,
       [101.76607 ,  53.15434 , 153.05981 , ...,  15.941224,  46.784527,
        102.77183 ],
       [ 31.027622,  40.749966,  45.443512, ...,  73.94004 ,  26.334076,
         78.63359 ],
       [154.06558 , 117.85822 , 176.19229 , ..., 112.82942 , 118.193474,
         88.02068 ]], shape=(217, 181), dtype=float32)

In [21]:
difference = np.abs(output_array - input_array)
print(f"Max difference: {np.max(difference)}")
print(f"Mean difference: {np.mean(difference)}")

Max difference: 804.9505615234375
Mean difference: 0.6459970474243164


In [17]:
from pathlib import Path
from typing import Union
import numpy as np
from PIL import Image

def adaptive_median_filter(
    input_file: Union[str, Path],
    output_file: Union[str, Path],
    max_radius: int
) -> None:
    # Leer la imagen usando Pillow y convertirla a un array de NumPy
    input_image = Image.open(input_file).convert("L")  # Convertir a escala de grises
    np_image = np.array(input_image, dtype=np.uint8)

    height, width = np_image.shape
    output_image = np.copy(np_image)

    for y in range(height):
        for x in range(width):
            output_image[y, x] = process_pixel(np_image, x, y, max_radius)

    # Convertir el array de salida nuevamente a una imagen y guardarla
    output_image = Image.fromarray(output_image)
    output_image.save(output_file)

def process_pixel(image: np.ndarray, x: int, y: int, max_radius: int) -> int:
    """Aplica el filtro de mediana adaptativo a un solo píxel"""
    radius = 1
    while radius <= max_radius:
        window = get_window(image, x, y, radius)
        min_val, med_val, max_val = np.min(window), np.median(window), np.max(window)

        # Nivel 1: Evaluar la mediana
        if min_val < med_val < max_val:
            # Nivel 2: Evaluar el valor del píxel original
            pixel_value = image[y, x]
            if min_val < pixel_value < max_val:
                return pixel_value
            else:
                return med_val
        else:
            radius += 1  # Aumentar el radio

    return image[y, x]  # Si se supera max_radius, devolver el píxel original

def get_window(image: np.ndarray, x: int, y: int, radius: int) -> np.ndarray:
    """Extrae la ventana 2D alrededor de un píxel, respetando los bordes de la imagen."""
    y_min, y_max = max(0, y - radius), min(image.shape[0], y + radius + 1)
    x_min, x_max = max(0, x - radius), min(image.shape[1], x + radius + 1)
    
    return image[y_min:y_max, x_min:x_max]

# Ejemplo de uso
adaptive_median_filter(BASE_PATH / "inputs/salt_pepper.png", "output.png", max_radius=1)


In [14]:
def adaptive_median_filter_2d(
    input_file: Union[str, Path],
    output_file: Union[str, Path],
    max_size: int = 9
) -> None:
    """
    2D version that more closely mimics the MATLAB implementation
    """
    # Read as 2D image if possible
    itk_image = itk.imread(str(input_file))
    np_image = itk.GetArrayFromImage(itk_image)
    
    # Handle case where image is 3D but we want to process as 2D
    if len(np_image.shape) == 3 and np_image.shape[0] == 1:
        np_image = np_image[0]  # Extract single slice
    elif len(np_image.shape) == 3:
        # Process each slice separately for true 3D images
        output_slices = []
        for z in range(np_image.shape[0]):
            slice_image = np_image[z]
            output_slice = process_image_2d(slice_image, max_size)
            output_slices.append(output_slice)
        output_image = np.stack(output_slices)
    else:
        # Process single 2D image
        output_image = process_image_2d(np_image, max_size)
    
    output_image = Image.fromarray(output_image)
    output_image.save(output_file)
    
    return np_image, output_image


def process_image_2d(image: np.ndarray, max_size: int = 9) -> np.ndarray:
    """Process a 2D image with adaptive median filter"""
    height, width = image.shape
    output_image = np.copy(image)
    
    for y in range(height):
        for x in range(width):
            output_image[y, x] = process_pixel_matlab_style_2d(image, x, y, max_size)
            
    return output_image


def process_pixel_matlab_style_2d(image: np.ndarray, x: int, y: int, max_size: int = 9) -> int:
    """2D version of the adaptive median filter process"""
    center_pixel = image[y, x]
    
    # Define window sizes to match MATLAB (3×3, 5×5, 7×7, 9×9)
    window_sizes = [3, 5, 7, 9]
    window_sizes = [size for size in window_sizes if size <= max_size]
    
    for size in window_sizes:
        radius = size // 2
        
        # Get 2D window
        y_min, y_max = max(0, y - radius), min(image.shape[0], y + radius + 1)
        x_min, x_max = max(0, x - radius), min(image.shape[1], x + radius + 1)
        window = image[y_min:y_max, x_min:x_max]
        
        # Skip if window is empty
        if window.size == 0:
            continue
            
        min_val = np.min(window)
        med_val = np.median(window)
        max_val = np.max(window)
        
        # Same decision rules as MATLAB
        if min_val < med_val < max_val:
            if min_val < center_pixel < max_val:
                return center_pixel
            else:
                return int(med_val)
    
    return center_pixel

In [15]:
adaptive_median_filter_2d(BASE_PATH / "inputs/salt_pepper2.png", "output.png")

(array([[[255, 255, 255, 255],
         [231, 231, 231, 255],
         [130, 130, 130, 255],
         ...,
         [252, 252, 252, 255],
         [254, 254, 254, 255],
         [255, 255, 255, 255]],
 
        [[255, 255, 255, 255],
         [230, 230, 230, 255],
         [123, 123, 123, 255],
         ...,
         [251, 251, 251, 255],
         [254, 254, 254, 255],
         [255, 255, 255, 255]],
 
        [[255, 255, 255, 255],
         [230, 230, 230, 255],
         [124, 124, 124, 255],
         ...,
         [251, 251, 251, 255],
         [254, 254, 254, 255],
         [255, 255, 255, 255]],
 
        ...,
 
        [[255, 255, 255, 255],
         [240, 240, 240, 255],
         [175, 175, 175, 255],
         ...,
         [243, 243, 243, 255],
         [252, 252, 252, 255],
         [255, 255, 255, 255]],
 
        [[255, 255, 255, 255],
         [240, 240, 240, 255],
         [178, 178, 178, 255],
         ...,
         [243, 243, 243, 255],
         [252, 252, 252, 255],
    